In [0]:
%run ../init/kafka_config

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType
from pyspark.sql.functions import col

In [0]:
dbutils.widgets.dropdown("action", "add_user",
    ["add_user", "add_item", "add_inventory", "view_all","restock_all"],
    "Select Action"
)

In [0]:
dbutils.widgets.dropdown("restock", "False", ["True", "False"], "Restock Mode")

In [0]:
# user fields
dbutils.widgets.text("user_id",  "",   "User ID")
dbutils.widgets.text("username", "",   "Username")
dbutils.widgets.text("email",    "",   "Email")
dbutils.widgets.text("balance",  "0",  "Balance")

# item fields
dbutils.widgets.text("item_id",   "",  "Item ID")
dbutils.widgets.text("item_name", "",  "Item Name")
dbutils.widgets.text("price",     "0", "Price")

# inventory fields
dbutils.widgets.text("quantity",  "0", "Quantity")

dbutils.widgets.text("restock_quantity", "10", "Restock All Quantity")

In [0]:
action    = dbutils.widgets.get("action")
user_id   = dbutils.widgets.get("user_id")
username  = dbutils.widgets.get("username")
email     = dbutils.widgets.get("email")
balance   = float(dbutils.widgets.get("balance") or 0)
item_id   = dbutils.widgets.get("item_id")
item_name = dbutils.widgets.get("item_name")
price     = float(dbutils.widgets.get("price") or 0)
quantity  = int(dbutils.widgets.get("quantity") or 0)
restock = dbutils.widgets.get("restock") == "True"
print(f"Action: {action}")

In [0]:
# ─────────────────────────────────────────
# FUNCTION 1 — Add User
# ─────────────────────────────────────────

def add_user(user_id, username, email, balance):
    existing = spark.read.table("users") \
        .filter(col("user_id") == user_id)

    if existing.count() > 0:
        print(f"✗ User {user_id} already exists — skipping")
        return

    if not user_id or not username or not email:
        print("✗ user_id, username and email are required")
        return

    if balance < 0:
        print("✗ Balance cannot be negative")
        return

    new_user = [(user_id, username, email, float(balance))]

    schema = StructType([
        StructField("user_id",  StringType(), nullable=False),
        StructField("username", StringType(), nullable=False),
        StructField("email",    StringType(), nullable=False),
        StructField("balance",  DoubleType(), nullable=False)
    ])

    spark.createDataFrame(new_user, schema) \
        .write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("users")

    print(f"✓ User added: {user_id} | {username} | {email} | balance: {balance}")



In [0]:
# ─────────────────────────────────────────
# FUNCTION 2 — Add Item
# ─────────────────────────────────────────

def add_item(item_id, item_name, price):
    existing = spark.read.table("items") \
        .filter(col("item_id") == item_id)

    if existing.count() > 0:
        print(f"✗ Item {item_id} already exists — skipping")
        return

    if not item_id or not item_name:
        print("✗ item_id and item_name are required")
        return

    if price <= 0:
        print("✗ Price must be greater than 0")
        return

    new_item = [(item_id, item_name, float(price))]

    schema = StructType([
        StructField("item_id",   StringType(), nullable=False),
        StructField("item_name", StringType(), nullable=False),
        StructField("price",     DoubleType(), nullable=False)
    ])

    spark.createDataFrame(new_item, schema) \
        .write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("items")

    print(f"✓ Item added: {item_id} | {item_name} | price: {price}")


In [0]:
# ─────────────────────────────────────────
# FUNCTION 3 — Add Inventory
# ─────────────────────────────────────────
def add_inventory(item_id, quantity, restock=False):
    """
    restock=False → set quantity to exact value
    restock=True  → add quantity to current stock
    """
    item_exists = spark.read.table("items") \
        .filter(col("item_id") == item_id)

    if item_exists.count() == 0:
        print(f"✗ Item {item_id} not found — add item first")
        return

    if quantity < 0:
        print("✗ Quantity cannot be negative")
        return

    inv_exists = spark.read.table("inventory") \
        .filter(col("item_id") == item_id)

    if inv_exists.count() > 0:
        if restock:
            # add to existing quantity
            current_qty = inv_exists.first()["quantity_available"]
            new_qty = current_qty + quantity
            print(f"Restocking: {item_id} | current: {current_qty} + {quantity} = {new_qty}")
        else:
            # set to exact value
            new_qty = quantity
            print(f"Setting: {item_id} → quantity = {new_qty}")

        DeltaTable.forName(spark, "inventory").update(
            condition = col("item_id") == item_id,
            set       = {"quantity_available": str(new_qty)}
        )
        print(f"✓ Inventory updated: {item_id} → {new_qty}")

    else:
        # new inventory row
        new_inv = [(item_id, int(quantity))]
        schema = StructType([
            StructField("item_id",            StringType(),  nullable=False),
            StructField("quantity_available", IntegerType(), nullable=False)
        ])
        spark.createDataFrame(new_inv, schema) \
            .write.format("delta").mode("append").saveAsTable("inventory")
        print(f"✓ Inventory added: {item_id} → quantity: {quantity}")

In [0]:
def restock_all(restock_quantity):
    """
    Adds restock_quantity to every item in inventory
    """
    items_df = spark.read.table("inventory")
    
    DeltaTable.forName(spark, "inventory").update(
        set = {"quantity_available": 
            f"quantity_available + {restock_quantity}"}
    )
    
    print(f"✓ All items restocked by +{restock_quantity}")
    spark.read.table("inventory").show(truncate=False)

In [0]:
# ─────────────────────────────────────────
# FUNCTION 4 — View All
# ─────────────────────────────────────────

def view_master_data():
    print("── Users ───────────────────────────────")
    spark.read.table("users").show(truncate=False)

    print("── Items ───────────────────────────────")
    spark.read.table("items").show(truncate=False)

    print("── Inventory ───────────────────────────")
    spark.read.table("inventory").show(truncate=False)


In [0]:
if action == "add_user":
    add_user(user_id, username, email, balance)

elif action == "add_item":
    add_item(item_id, item_name, price)

elif action == "add_inventory":
    add_inventory(item_id, quantity)
elif action == "add_inventory":
    add_inventory(item_id, quantity, restock)
elif action == "restock_all":
    restock_quantity = int(dbutils.widgets.get("restock_quantity"))
    restock_all(restock_quantity)
elif action == "view_all":
    view_master_data()


else:
    print(f"✗ Unknown action: {action}")